# 3.6 — Regression Metrics: MAE, MSE, RMSE, R²

For regression problems (predicting a number), accuracy makes no sense. You measure how far off your predictions are.

| Metric | Formula | Units | Use when |
|--------|---------|-------|----------|
| MAE | avg\|actual-pred\| | Same as target | Easy to explain, outliers present |
| MSE | avg(actual-pred)² | Squared units | Intermediate step to RMSE |
| RMSE | √MSE | Same as target | Big errors are especially bad |
| R² | 1-(SS_res/SS_tot) | 0 to 1 (unitless) | How good is the model overall? |

> **Key rule:** Always report RMSE and R² together. RMSE = how big the errors are. R² = how good the model is in context.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# House price dataset
data = {
    'size_sqft':  [600, 800, 1000, 1200, 1500, 1800, 2000, 2200, 2500, 3000],
    'bedrooms':   [1,   2,   2,    3,    3,    4,    4,    5,    5,    6],
    'price_lakh': [35,  48,  55,   65,   78,   92,   105,  118,  135,  165]
}
df = pd.DataFrame(data)
print(df)

## Understanding Errors First

In [ ]:
# Simple example — 5 houses, manual errors
actual    = np.array([50, 80, 60, 90, 70])
predicted = np.array([52, 75, 58, 100, 69])

errors = actual - predicted
print("Actual:   ", actual)
print("Predicted:", predicted)
print("Errors:   ", errors)
print("\nPositive error = we under-predicted")
print("Negative error = we over-predicted")

## MAE — Mean Absolute Error

Average of absolute errors. Same units as target. Easy to interpret.

In [ ]:
# Manual calculation
abs_errors = np.abs(errors)
mae_manual = abs_errors.mean()
print("Absolute errors:", abs_errors)
print(f"MAE (manual):   {mae_manual:.2f}")

# sklearn
mae_sklearn = mean_absolute_error(actual, predicted)
print(f"MAE (sklearn):  {mae_sklearn:.2f}")
print(f"\nInterpretation: On average, predictions are off by ₹{mae_manual:.1f} lakhs")

## MSE — Mean Squared Error

Average of squared errors. Penalises big errors heavily. Units are squared (not intuitive).

In [ ]:
# Manual calculation
squared_errors = errors ** 2
mse_manual = squared_errors.mean()
print("Squared errors:", squared_errors)
print(f"MSE (manual):   {mse_manual:.2f}")
print(f"MSE (sklearn):  {mean_squared_error(actual, predicted):.2f}")
print(f"\nNote: error of -10 → squared to 100 (much larger than MAE's 10)")
print("Big errors dominate MSE — that's the point")

## RMSE — Root Mean Squared Error

Square root of MSE. Same units as target AND penalises big errors. Most commonly used regression metric.

In [ ]:
rmse_manual = np.sqrt(mse_manual)
print(f"MSE:  {mse_manual:.2f}")
print(f"RMSE: {rmse_manual:.2f}")
print(f"MAE:  {mae_manual:.2f}")
print(f"\nRMSE ({rmse_manual:.2f}) > MAE ({mae_manual:.2f})")
print("The gap exists because RMSE penalises the big -10 error more than MAE does")

## R² — Coefficient of Determination

How much better is your model vs just predicting the mean for everyone?

`R² = 1 - (SS_res / SS_tot)`

In [ ]:
# Manual R² calculation
mean_actual = actual.mean()
ss_res = np.sum((actual - predicted) ** 2)  # your model's total error
ss_tot = np.sum((actual - mean_actual) ** 2) # dumb baseline's total error

r2_manual = 1 - (ss_res / ss_tot)

print(f"Mean of actual values: {mean_actual}")
print(f"SS_res (your model):   {ss_res}")
print(f"SS_tot (baseline):     {ss_tot}")
print(f"R² = 1 - ({ss_res}/{ss_tot}) = {r2_manual:.3f}")
print(f"R² (sklearn):          {r2_score(actual, predicted):.3f}")
print(f"\nModel explains {r2_manual*100:.1f}% of the variation in prices")

## Full Example — House Price Prediction

In [ ]:
X = df[['size_sqft', 'bedrooms']]
y = df['price_lakh']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred)

print(f"MAE:  {mae:.2f}  ← avg off by ₹{mae:.1f}L")
print(f"MSE:  {mse:.2f}  ← squared units")
print(f"RMSE: {rmse:.2f}  ← off by ₹{rmse:.1f}L (penalises big errors)")
print(f"R²:   {r2:.3f}  ← model explains {r2*100:.1f}% of price variation")

In [ ]:
# Actual vs predicted table
comparison = pd.DataFrame({
    'actual':    y_test.values,
    'predicted': y_pred.round(1),
    'error':     (y_test.values - y_pred).round(1),
    'abs_error': np.abs(y_test.values - y_pred).round(1)
})
print(comparison)

In [ ]:
# Plot actual vs predicted
plt.figure(figsize=(6, 5))
plt.scatter(y_test, y_pred, color='#378ADD', s=80)
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         'k--', label='Perfect prediction line')
plt.xlabel('Actual price (₹L)')
plt.ylabel('Predicted price (₹L)')
plt.title(f'Actual vs Predicted — R²={r2:.3f}')
plt.legend()
plt.tight_layout()
plt.show()
print("Points close to the diagonal = good predictions")

## MAE vs RMSE — Outlier Effect

In [ ]:
# Compare MAE vs RMSE with and without a large error
actual_clean  = np.array([50, 60, 70, 80, 90])
pred_clean    = np.array([52, 62, 68, 82, 88])  # consistent small errors
pred_outlier  = np.array([52, 62, 68, 82, 40])  # one huge error

print("Consistent errors:")
print(f"  MAE:  {mean_absolute_error(actual_clean, pred_clean):.2f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(actual_clean, pred_clean)):.2f}")
print(f"  Gap:  {np.sqrt(mean_squared_error(actual_clean, pred_clean)) - mean_absolute_error(actual_clean, pred_clean):.2f}")

print("\nOne huge outlier error (90 predicted as 40):")
print(f"  MAE:  {mean_absolute_error(actual_clean, pred_outlier):.2f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(actual_clean, pred_outlier)):.2f}")
print(f"  Gap:  {np.sqrt(mean_squared_error(actual_clean, pred_outlier)) - mean_absolute_error(actual_clean, pred_outlier):.2f}")
print("\nLarge RMSE-MAE gap = your model has big outlier errors somewhere")

## When to Use Which

| Metric | Use when |
|--------|----------|
| MAE | Easy to explain; outliers are present and shouldn't dominate |
| MSE | Intermediate — rarely reported directly |
| RMSE | Default choice; large errors are especially bad |
| R² | Understanding overall model quality; comparing models |

> **Always report RMSE + R² together.** RMSE tells how big your errors are. R² tells how good the model is relative to the simplest possible baseline.

## Practice Task

In [ ]:
practice_data = {
    'study_hours': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'score':       [40, 45, 55, 58, 65, 70, 75, 82, 88, 95]
}
practice_df = pd.DataFrame(practice_data)
X_p = practice_df[['study_hours']]
y_p = practice_df['score']
X_tr, X_te, y_tr, y_te = train_test_split(X_p, y_p, test_size=0.3, random_state=42)

# Q1 — Train LinearRegression and calculate MAE, MSE, RMSE, R²
# YOUR CODE HERE

# Q2 — Calculate MAE manually (without sklearn)
# YOUR CODE HERE

# Q3 — Why is RMSE always >= MAE?
# ANSWER:

# Q4 — What does your R² score mean for this model?
# ANSWER:

# Q5 — To report to a teacher, which metric would you choose and why?
# ANSWER: